# PetenFire — Entrenamiento M1 en Google Colab

**Dataset:** `m1_dataset.parquet` | **Modelo:** LightGBM (probabilidades RAW, sin calibración isotónica)

**Subsampleo 10:1** para Colab Free (~960 K filas de 136 M totales)

---

## Instrucciones previas

1. Sube `m1_dataset.parquet` a Google Drive en la ruta: `Mi unidad/petenfire/data/`
2. Activa GPU: `Entorno de ejecución → Cambiar tipo de entorno → GPU T4 (gratis)`
3. Ejecuta **Run All** (`Ctrl+F9`)

---

### Splits temporales
| Split | Años | Filas aprox. |
|-------|------|--------------|
| train | 2018–2022 | 97 M |
| val   | 2023      | 19 M |
| test  | 2024      | 19 M |

### Criterio de éxito
| Métrica | Mínimo |
|---------|--------|
| AUC-PR (val) | > 0.10 |
| F1 (val, umbral óptimo) | > 0.20 |
| best_iteration_ | > 1 |

> **Nota:** El umbral de decisión se determina automáticamente buscando el máximo F1
> en la curva precision-recall de val (Celda 8). El valor se guarda en el artefacto
> y es el que debe usarse en inferencia.


In [ ]:
# Celda 2 — Instalar dependencias
# lightgbm, scikit-learn y joblib suelen venir en Colab; forzamos versiones estables
!pip install -q lightgbm scikit-learn joblib pandas pyarrow numpy
print("Dependencias instaladas.")

In [ ]:
# Celda 3 — Montar Google Drive
import os

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

# Ajusta estas rutas si guardaste el archivo en otra carpeta
DATASET_PATH = "/content/drive/MyDrive/petenfire/data/m1_dataset.parquet"
OUTPUT_DIR = "/content/drive/MyDrive/petenfire/models/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verificar que el dataset existe antes de continuar
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"No se encontró el dataset en: {DATASET_PATH}\n"
        "Asegúrate de haberlo subido a Google Drive en la carpeta "
        "'Mi unidad/petenfire/data/' y de haber montado el Drive correctamente."
    )

print(f"Dataset encontrado: {DATASET_PATH}")
print(f"Directorio de salida: {OUTPUT_DIR}")

In [ ]:
# Celda 4 — Cargar datos con subsampleo RAM-eficiente
#
# Estrategia:
#   - Leer el parquet en batches de 5 M filas para evitar picos de RAM
#   - train: todos los positivos + muestra 10:1 de negativos (seed=42)
#   - val / test: subsampleo a máx 500 K filas (proporcional, estratificado)

import gc

import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq

RANDOM_SEED = 42
SUBSAMPLE_RATIO = 10           # negativos por cada positivo en train
VAL_TEST_MAX_ROWS = 500_000    # máximo de filas para val y test
BATCH_SIZE = 5_000_000         # filas por batch de lectura

FEATURE_COLS = [
    # Clima
    "T2M", "RH2M", "WS10M", "PRECTOTCORR",
    # FWI
    "fwi", "ffmc_val", "dmc_val", "dc_val", "isi_val", "bui_val",
    # Precipitación acumulada
    "prec_acc7d", "prec_acc14d",
    # Vegetación
    "ndvi", "ndvi_lag7", "ndvi_lag14",
    # Topografía
    "elevation_m", "slope_deg", "aspect_deg",
    # Antrópico
    "dist_roads_km", "dist_settlements_km", "is_protected_area",
    # Temporal
    "month", "day_of_year",
]
TARGET_COL = "fire_occurred"
COLS_NEEDED = FEATURE_COLS + [TARGET_COL, "split"]


def ram_gb() -> float:
    """RAM usada por el proceso actual en GB."""
    return psutil.Process().memory_info().rss / 1e9


def subsample_split(
    pf: pq.ParquetFile,
    split_name: str,
    ratio: int = 10,
    max_rows: int | None = None,
    seed: int = 42,
) -> pd.DataFrame:
    """Lee el parquet en batches y retorna un DataFrame subsampled para el split dado.

    Para 'train': recoge todos los positivos + ratio:1 negativos aleatorios.
    Para 'val'/'test': muestrea max_rows filas manteniendo la proporción de positivos.
    """
    rng = np.random.default_rng(seed)
    pos_list: list[pd.DataFrame] = []
    neg_list: list[pd.DataFrame] = []

    print(f"  Leyendo split '{split_name}' en batches de {BATCH_SIZE:,} filas...")
    for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=COLS_NEEDED):
        df_batch = batch.to_pandas()
        df_batch = df_batch[df_batch["split"] == split_name]
        if df_batch.empty:
            continue

        pos_batch = df_batch[df_batch[TARGET_COL] == 1]
        neg_batch = df_batch[df_batch[TARGET_COL] == 0]

        pos_list.append(pos_batch)

        if split_name == "train":
            # Subsampleo proporcional al número de positivos en este batch
            n_neg_keep = min(len(neg_batch), len(pos_batch) * ratio)
            if n_neg_keep > 0:
                idx = rng.choice(len(neg_batch), size=n_neg_keep, replace=False)
                neg_list.append(neg_batch.iloc[idx])
        else:
            neg_list.append(neg_batch)

    all_pos = pd.concat(pos_list, ignore_index=True) if pos_list else pd.DataFrame()
    all_neg = pd.concat(neg_list, ignore_index=True) if neg_list else pd.DataFrame()

    if split_name != "train" and max_rows is not None:
        # Para val/test: subsamplear al total deseado manteniendo el ratio original
        total = len(all_pos) + len(all_neg)
        if total > max_rows:
            frac = max_rows / total
            n_pos = min(len(all_pos), max(1, int(len(all_pos) * frac)))
            n_neg = max_rows - n_pos
            idx_pos = rng.choice(len(all_pos), size=n_pos, replace=False)
            idx_neg = rng.choice(len(all_neg), size=min(n_neg, len(all_neg)), replace=False)
            all_pos = all_pos.iloc[idx_pos]
            all_neg = all_neg.iloc[idx_neg]

    result = pd.concat([all_pos, all_neg], ignore_index=True)
    result = result.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return result


# Verificar columnas disponibles antes de cargar todo
pf = pq.ParquetFile(DATASET_PATH)
schema_names = set(pf.schema_arrow.names)
available_features = [c for c in FEATURE_COLS if c in schema_names]
missing_features = [c for c in FEATURE_COLS if c not in schema_names]

if missing_features:
    print(f"AVISO: Features no encontradas en el parquet (se omitirán): {missing_features}")
print(f"Features disponibles: {len(available_features)}/{len(FEATURE_COLS)}")

# Ajustar COLS_NEEDED con solo las columnas que existen
COLS_NEEDED = available_features + [TARGET_COL, "split"]

print(f"\nRAM antes de carga: {ram_gb():.2f} GB")
print("Cargando train...")
train_df = subsample_split(pf, "train", ratio=SUBSAMPLE_RATIO, seed=RANDOM_SEED)
print(f"  train: {len(train_df):,} filas | positivos: {train_df[TARGET_COL].mean()*100:.2f}%")

print("Cargando val...")
val_df = subsample_split(pf, "val", max_rows=VAL_TEST_MAX_ROWS, seed=RANDOM_SEED)
print(f"  val:   {len(val_df):,} filas | positivos: {val_df[TARGET_COL].mean()*100:.4f}%")

print("Cargando test...")
test_df = subsample_split(pf, "test", max_rows=VAL_TEST_MAX_ROWS, seed=RANDOM_SEED)
print(f"  test:  {len(test_df):,} filas | positivos: {test_df[TARGET_COL].mean()*100:.4f}%")

print(f"\nRAM después de carga: {ram_gb():.2f} GB")
gc.collect()

In [ ]:
# Celda 5 — Preparar arrays numpy

import numpy as np

X_train = train_df[available_features].values.astype(np.float32)
y_train = train_df[TARGET_COL].values.astype(np.int32)

X_val = val_df[available_features].values.astype(np.float32)
y_val = val_df[TARGET_COL].values.astype(np.int32)

X_test = test_df[available_features].values.astype(np.float32)
y_test = test_df[TARGET_COL].values.astype(np.int32)

# Liberar DataFrames para recuperar RAM
del train_df, val_df, test_df
gc.collect()

print("Shapes y ratios de positivos:")
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"  {name:6s}: X={X.shape} | y positivos={y.sum():,} ({y.mean()*100:.3f}%)")

print(f"\nRAM después de conversión a numpy: {ram_gb():.2f} GB")

In [ ]:
# Celda 6 — Entrenar LightGBM
#
# scale_pos_weight = 1.0 porque el subsampleo ya balanceó las clases (~10:1)
# eval_metric = 'average_precision' (AUC-PR, apropiado para clases raras)
# early_stopping(30): para si no mejora en 30 rondas consecutivas

import time

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

N_ESTIMATORS = 500
LEARNING_RATE = 0.05
MAX_DEPTH = -1          # sin límite de profundidad (LightGBM lo controla con num_leaves)
THRESHOLD = 0.05        # umbral para métricas discretas (F1, precision, recall)

lgbm_model = LGBMClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    scale_pos_weight=1.0,   # subsampleo ya equilibró el dataset
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=-1,
)

print("Iniciando entrenamiento LightGBM...")
t0 = time.time()

lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    feature_name=available_features,
    callbacks=[
        early_stopping(30, verbose=False),
        log_evaluation(50),
    ],
)

elapsed = time.time() - t0
print(f"\nEntrenamiento completado en {elapsed/60:.1f} min")
print(f"Mejor iteración: {lgbm_model.best_iteration_}")

if lgbm_model.best_iteration_ <= 1:
    print("ADVERTENCIA: best_iteration_ = 1. El modelo probablemente no aprendió.")
    print("  → Verifica que el dataset tenga positivos y que las features no sean todas NaN.")

In [ ]:
# Celda 7 — Probabilidades RAW del modelo (sin calibración isotónica)
#
# La calibración isotónica entrenada sobre val COMPLETO (prevalencia ~0.064%)
# aprende a mapear scores de ~9% → ~0.064%, comprimiendo todo por debajo de
# cualquier umbral fijo útil. Resultado: F1=0 aunque AUC-ROC sea excelente.
#
# Solución: usar los scores RAW de LightGBM y encontrar el umbral óptimo en val.

import numpy as np
import pandas as pd

# Pasar DataFrames con nombres de columnas para evitar el warning
# "X does not have valid feature names" (modelo entrenado con feature_name=)
X_val_df  = pd.DataFrame(X_val,  columns=available_features)
X_test_df = pd.DataFrame(X_test, columns=available_features)

# Probabilidades RAW directamente del LightGBM
raw_proba_val  = lgbm_model.predict_proba(X_val_df)[:, 1]
raw_proba_test = lgbm_model.predict_proba(X_test_df)[:, 1]

print(f"Probabilidades RAW val  — min={raw_proba_val.min():.6f}, "
      f"max={raw_proba_val.max():.6f}, media={raw_proba_val.mean():.6f}")
print(f"Probabilidades RAW test — min={raw_proba_test.min():.6f}, "
      f"max={raw_proba_test.max():.6f}, media={raw_proba_test.mean():.6f}")
print("\nNota: calibración isotónica deshabilitada (comprimía los scores al rango "
      "de prevalencia real y rompía el F1).")


In [ ]:
# Celda 8 — Búsqueda de umbral óptimo y evaluación
#
# 1. Precision-Recall curve en val → umbral que maximiza F1.
# 2. Se aplica ese mismo umbral (fijado en val) a test. No re-optimizar en test.

import numpy as np
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

# ── Paso 1: encontrar umbral óptimo en val ────────────────────────────────────
precisions, recalls, thresholds_pr = precision_recall_curve(y_val, raw_proba_val)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = int(f1_scores[:-1].argmax())
optimal_threshold = float(thresholds_pr[best_idx])
best_f1_val = float(f1_scores[best_idx])

print("Búsqueda de umbral óptimo en val (curva precision-recall):")
print(f"  Umbral óptimo    : {optimal_threshold:.6f}")
print(f"  F1 máximo en val : {best_f1_val:.4f}")
print(f"  Precisión        : {precisions[best_idx]:.4f}")
print(f"  Recall           : {recalls[best_idx]:.4f}")


# ── Paso 2: función de evaluación ─────────────────────────────────────────────
def evaluate_split(
    proba: np.ndarray,
    y: np.ndarray,
    split_name: str,
    threshold: float,
) -> dict:
    """Calcula métricas completas para un split dado un array de probabilidades."""
    pred = (proba >= threshold).astype(int)

    metrics = {
        "auc_roc":   float(roc_auc_score(y, proba)),
        "auc_pr":    float(average_precision_score(y, proba)),
        "f1":        float(f1_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall":    float(recall_score(y, pred, zero_division=0)),
        "threshold": threshold,
    }

    print(f"\n{'='*55}")
    print(f"  {split_name.upper()} | threshold={threshold:.6f}")
    print(f"{'='*55}")
    print(f"  AUC-ROC   : {metrics['auc_roc']:.4f}")
    print(f"  AUC-PR    : {metrics['auc_pr']:.4f}")
    print(f"  F1        : {metrics['f1']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    return metrics


# ── Paso 3: evaluar con el umbral óptimo (fijado en val, aplicado a test) ────
val_metrics  = evaluate_split(raw_proba_val,  y_val,  "VAL",  optimal_threshold)
test_metrics = evaluate_split(raw_proba_test, y_test, "TEST", optimal_threshold)

# ── Paso 4: verificar criterios de éxito ──────────────────────────────────────
auc_pr_val = val_metrics["auc_pr"]

print("\n" + "="*55)
print("  CRITERIOS DE ÉXITO")
print("="*55)
print(f"  {'[OK]' if auc_pr_val > 0.10 else '[!!]'} AUC-PR val > 0.10: "
      f"{'PASA' if auc_pr_val > 0.10 else 'FALLA'}")
print(f"  {'[OK]' if best_f1_val > 0.20 else '[!!]'} F1 val (umbral óptimo) > 0.20: "
      f"{'PASA' if best_f1_val > 0.20 else 'FALLA'}")
print(f"  {'[OK]' if lgbm_model.best_iteration_ > 1 else '[!!]'} best_iteration_ > 1: "
      f"{'PASA' if lgbm_model.best_iteration_ > 1 else 'FALLA'}")
print(f"  [INFO] Umbral óptimo encontrado: {optimal_threshold:.6f}")

all_pass = (auc_pr_val > 0.10) and (best_f1_val > 0.20) and (lgbm_model.best_iteration_ > 1)
print("="*55)
if all_pass:
    print("  >> MODELO LISTO PARA GUARDAR <<")
else:
    print("  >> REVISA LOS CRITERIOS FALLIDOS ANTES DE REPORTAR <<")
print("="*55)


In [ ]:
# Celda 9 — Guardar modelo
#
# Artefacto en formato dict para facilitar inspección sin reimportar clases.
# threshold = umbral óptimo encontrado en val (no 0.05 fijo).
# calibration = 'none' — se usan probabilidades RAW del LightGBM.

import joblib

artifact = {
    # Objeto del modelo
    "model": lgbm_model,
    # Configuración de inferencia
    "feature_names": available_features,
    "threshold": optimal_threshold,   # umbral aprendido de val, no 0.05 fijo
    "calibration": "none",            # sin calibración isotónica
    # Métricas de evaluación
    "metrics": {
        "val": val_metrics,
        "test": test_metrics,
    },
    # Metadatos del entrenamiento
    "subsample_ratio": SUBSAMPLE_RATIO,
    "random_seed": RANDOM_SEED,
    "best_iteration": lgbm_model.best_iteration_,
    "train_split": "2018-2022",
    "val_split": "2023",
    "test_split": "2024",
}

model_path = f"{OUTPUT_DIR}/m1_lightgbm_colab.joblib"
joblib.dump(artifact, model_path, compress=3)   # compress=3 reduce tamaño ~40%

print(f"Modelo guardado en: {model_path}")
print(f"Umbral guardado  : {optimal_threshold:.6f}")
print(f"Calibración      : none (probabilidades RAW)")

# Mostrar tamaño del archivo
size_mb = os.path.getsize(model_path) / 1e6
print(f"Tamaño del artefacto: {size_mb:.1f} MB")


## Después de entrenar

### 1. Descarga el artefacto
Desde Google Drive, descarga `m1_lightgbm_colab.joblib` (ruta: `petenfire/models/`).

### 2. Cópialo al proyecto local
```bash
cp ~/Downloads/m1_lightgbm_colab.joblib <proyecto>/models_artifacts/m1/
```

### 3. Reporta las métricas al orquestador

Indica los siguientes valores de la celda 8:

| Métrica | Valor val | Valor test |
|---------|-----------|------------|
| AUC-PR  | _rellenar_ | _rellenar_ |
| AUC-ROC | _rellenar_ | _rellenar_ |
| F1 (umbral óptimo) | _rellenar_ | _rellenar_ |
| Umbral óptimo | _rellenar_ | — |
| best_iteration_ | _rellenar_ | — |

### 4. Si algún criterio falla
- **AUC-PR ≤ 0.10**: las features pueden tener demasiados NaN. Ejecuta la celda 5
  e inspecciona `np.isnan(X_train).mean(axis=0)`.
- **F1 ≤ 0.20 (umbral óptimo)**: el modelo no discrimina lo suficiente. Revisa
  la distribución de features en train vs val. Prueba aumentar `N_ESTIMATORS` a 1000
  o reducir `LEARNING_RATE` a 0.02.
- **best_iteration_ = 1**: el modelo no aprendió. Verifica que `y_train.sum() > 0`
  y que `X_train` no tenga todas las columnas en NaN.

### 5. Inferencia con el artefacto guardado
```python
import joblib, pandas as pd

art = joblib.load("models_artifacts/m1/m1_lightgbm_colab.joblib")
model     = art["model"]
threshold = art["threshold"]   # umbral óptimo aprendido de val
features  = art["feature_names"]

X_new = pd.DataFrame(new_data, columns=features)
proba = model.predict_proba(X_new)[:, 1]
pred  = (proba >= threshold).astype(int)
```
